In [7]:
from ingest import load_faq_data, build_index_sqlite
from sqlitesearch import TextSearchIndex
from pprint import pprint

In [2]:
# load the faq data and subset just for the llm zoomcamp faqs
documents = load_faq_data()
docs_llm = [doc for doc in documents if doc['course']=='llm-zoomcamp']
len(docs_llm)

85

In [ ]:
# create the index and store in faq.db file on disk
build_index_sqlite(docs_llm)

Added 10 documents...
Added 20 documents...
Added 30 documents...
Added 40 documents...
Added 50 documents...
Added 60 documents...
Added 70 documents...
Added 80 documents...
Done. Index saved to faq.db


In [5]:
# connect to faq.db and check the number of documents
sqlite_index = TextSearchIndex(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course'],
    db_path='faq.db'
)

sqlite_index.count()

85

In [8]:
# try a search
results = sqlite_index.search(query='Can I still join the course after it started?', num_results=5)
pprint([doc['question'] for doc in results])

['I just discovered the course. Can I still join?',
 'How do I start using Google Gemini models in the Module 1 notebook through '
 'the OpenAI-compatible endpoint?',
 'I missed the first homework - can I still get a certificate?',
 'Homework: Why does the content keep changing?',
 'What happens to code saved in Codespaces if I do not commit it?']


In [ ]:
# use the sqlite_index with RAGBase; now we do not execute the faq data loading and index building steps as we are using persistent storage
from rag_helper import RAGBase
from openai import OpenAI
from dotenv import load_dotenv

openai_client = OpenAI()
load_dotenv()

# we swap the index with 'sqlite_index' instead of 'index' in 05-refactor-rag.ipynb; this works because sqlitesearch follows the same API as minsearch - both have 
# a search method that takes a query, boost_dict, filter_dict, and num_results; if the API were different, we'd need to subclass RAGBase and override the search 
# method to adapt to the new backend.
assistant = RAGBase(
    index=sqlite_index,
    llm_client=openai_client
)

In [ ]:
# try with a query
answer = assistant.rag("Can I still join the course after it started?")
print(answer)

Yes, you can still join the course after it has started. However, to receive a certificate, you must submit your project while submissions are still being accepted.


In [ ]:
# close the db connection
sqlite_index.close()